### GPT with MoE (MixGPT)  
(Can switch between MoE and Dense mode)  

In [10]:
import os
from dataclasses import dataclass, field
from typing import Optional, Tuple
import torch


# -----------------------------------------------------------------------------
# Configuration & Paths
# -----------------------------------------------------------------------------

@dataclass
class ProjectPaths:
    """Centralized configuration for absolute file paths."""
    base_dir: str = os.getcwd() #"./" # Change this to your root production directory

    @property
    def tokenizer_dir(self) -> str:
        return os.path.join(self.base_dir, "custom_tokenizer")

    @property
    def model_checkpoints_dir(self) -> str:
        return os.path.join(self.base_dir, "trained_models")

    @property
    def data_corpus(self) -> str:
        return os.path.join(self.base_dir, "data", "corpus.txt")


@dataclass
class MoEArgs:
    """Configuration specific to the Mixture of Experts layers."""
    num_experts: int = 2
    top_k_experts: int = 1
    z_loss_coef: float = 0.001
    moe_loss_coef: float = 0.01
    hidden_dim_multplier_perexp: int = 2


@dataclass
class ModelArgs:
    """Unified configuration for the Transformer architecture."""
    vocab_size: int
    d_model: int
    num_heads: int
    num_layers: int
    org_context_length: int = 1024
    scale_factor: float = 1.0
    rotary_freq_base: int = 10000
    dropout: float = 0.1
    hidden_dim_multiplier: int = 4
    moe_args: Optional[MoEArgs] = None
    device: torch.device = field(default_factory=lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu"))

    def __post_init__(self):
        assert self.d_model % self.num_heads == 0, "d_model must be divisible by num_heads"

# -----------------------------------------------------------------------------
# Training Configuration
# -----------------------------------------------------------------------------

@dataclass
class TrainingArgs:
    """Strictly training-loop specific parameters."""
    tot_steps: int = 500
    batch_size: int = 2
    grad_accum_steps: int = 1
    lr_rate: float = 0.0005
    grad_clip: float = 1.0



In [ ]:
# Model

import math
from typing import Optional, Tuple
import torch.nn as nn
import torch.nn.functional as F
import time
import yaml
from transformers import AutoTokenizer

# -----------------------------------------------------------------------------
# Feed-Forward Networks
# -----------------------------------------------------------------------------

class FeedForwardSwiGLU(nn.Module):
    """Standard SwiGLU FeedForward network for non-MoE layers."""
    def __init__(self, d_model: int, hidden_dim: int, bias: bool = False):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden_dim, bias=bias)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=bias)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class FeedForward_Gelu(nn.Module):
    """Standard GELU FeedForward network for non-MoE layers."""
    def __init__(self, emb_dim:int, hidden_dim:int, bias=False):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim, bias),
            nn.GELU(),
            nn.Linear(hidden_dim, emb_dim, bias),
        )

    def forward(self, x):
        return self.layers(x)

class MoEFeedForward(nn.Module):
    """Mixture of Experts layer with Z-Loss, Load Balancing, and No zero-padding optimization."""
    def __init__(self, moe_args: MoEArgs, emb_dim: int, hidden_dim: int):
        super().__init__()
        self.num_experts = moe_args.num_experts
        self.num_experts_per_tok = moe_args.top_k_experts
        self.emb_dim = emb_dim
        self.hidden_dim = hidden_dim

        self.z_loss_coef = moe_args.z_loss_coef
        self.moe_loss_coef = moe_args.moe_loss_coef

        self.gate = nn.Linear(emb_dim, self.num_experts, bias=False)

        # Batched weights for fast expert computation: [num_experts, in_features, out_features]
        self.w1 = nn.Parameter(torch.empty(self.num_experts, emb_dim, hidden_dim))
        self.w2 = nn.Parameter(torch.empty(self.num_experts, emb_dim, hidden_dim))
        self.w3 = nn.Parameter(torch.empty(self.num_experts, hidden_dim, emb_dim))

        self._init_weights()

    def _init_weights(self):
        """Manually calculate bounds to match nn.Linear PyTorch initialization exactly."""
        bound_w12 = 1 / math.sqrt(self.emb_dim)
        nn.init.uniform_(self.w1, -bound_w12, bound_w12)
        nn.init.uniform_(self.w2, -bound_w12, bound_w12)

        bound_w3 = 1 / math.sqrt(self.hidden_dim)
        nn.init.uniform_(self.w3, -bound_w3, bound_w3)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        batch, seq_len, emb_dim = x.shape
        x_flat = x.view(-1, emb_dim)

        # 1. Routing & Z-Loss
        router_logits = self.gate(x_flat)
        z_loss = torch.mean(torch.logsumexp(router_logits, dim=-1) ** 2)

        routing_probs = F.softmax(router_logits, dim=-1)
        routing_weights, selected_experts = torch.topk(routing_probs, self.num_experts_per_tok, dim=-1)
        routing_weights = routing_weights / routing_weights.sum(dim=-1, keepdim=True)

        # 2. Auxiliary Load Balancing Loss
        P_i = routing_probs.mean(dim=0)
        with torch.no_grad():
            tokens_per_expert = torch.bincount(selected_experts.view(-1), minlength=self.num_experts)
            f_i = tokens_per_expert.float() / selected_experts.numel()

        load_balancing_loss = self.num_experts * torch.sum(f_i * P_i)
        aux_loss = (self.moe_loss_coef * load_balancing_loss) + (self.z_loss_coef * z_loss)

        # 3. Fast Expert Computation
        flat_expert_indices = selected_experts.view(-1)
        flat_routing_weights = routing_weights.view(-1)
        flat_token_indices = torch.arange(x_flat.size(0), device=x.device).repeat_interleave(self.num_experts_per_tok)

        out_flat = torch.zeros_like(x_flat)

        for i in range(self.num_experts):
            expert_mask = (flat_expert_indices == i)
            if not expert_mask.any():
                continue

            # Extract only valid tokens and weights
            valid_token_indices = flat_token_indices[expert_mask]
            curr_tokens = x_flat[valid_token_indices]
            curr_weights = flat_routing_weights[expert_mask].unsqueeze(-1)

            # Dense Matrix Multiplications for specific expert
            h1 = torch.matmul(curr_tokens, self.w1[i])
            h2 = torch.matmul(curr_tokens, self.w2[i])
            hidden = F.silu(h1) * h2
            expert_out = torch.matmul(hidden, self.w3[i])

            # Scale and accumulate
            expert_out = expert_out * curr_weights
            out_flat = out_flat.index_add(0, valid_token_indices, expert_out)

        return out_flat.view(batch, seq_len, emb_dim), aux_loss

# -----------------------------------------------------------------------------
# Attention Mechanism
# -----------------------------------------------------------------------------

class Attention(nn.Module):
    """Multi-head attention utilizing YaRN/RoPE """
    def __init__(self, args: ModelArgs, qkv_bias: bool = False):
        super().__init__()
        self.num_heads = args.num_heads
        self.head_dim = args.d_model // args.num_heads
        self.model_dim = args.d_model
        self.org_context_length = args.org_context_length
        self.device = args.device

        assert self.head_dim % 2 == 0, "head_dim must be even for RoPE"

        # YaRN parameter setup
        self.extended_context_length = int(args.org_context_length * args.scale_factor)
        self.scale_factor = args.scale_factor if args.scale_factor else (self.extended_context_length / args.org_context_length)

        # QKV Projections
        self.W_query = nn.Linear(args.d_model, args.d_model, bias=qkv_bias)
        self.W_key = nn.Linear(args.d_model, args.d_model, bias=qkv_bias)
        self.W_value = nn.Linear(args.d_model, args.d_model, bias=qkv_bias)
        self.W_out = nn.Linear(args.d_model, args.d_model, bias=qkv_bias)

        self.dropout = nn.Dropout(args.dropout)

        # QK Normalization (Stable Training)
        self.q_norm = nn.RMSNorm(self.head_dim, eps=1e-6)
        self.k_norm = nn.RMSNorm(self.head_dim, eps=1e-6)

        # Precompute rotational frequencies
        self.cos, self.sin = self._precompute_yarn_freqs(base=args.rotary_freq_base)

        # KV Cache for inference
        self.cache_k = None
        self.cache_v = None

    def _init_cache(self, batch_size: int, device: torch.device, dtype: torch.dtype):
        self.cache_k = torch.empty(batch_size, self.num_heads, self.extended_context_length, self.head_dim, device=device, dtype=dtype)
        self.cache_v = torch.empty(batch_size, self.num_heads, self.extended_context_length, self.head_dim, device=device, dtype=dtype)

    def _precompute_yarn_freqs(self, base: float = 10000.0, beta: int = 32, alpha: int = 1):
        """Calculates YaRN interpolated/extrapolated frequencies."""
        pos_freqs = base ** (torch.arange(0, self.head_dim, 2, dtype=torch.float, device=self.device) / self.head_dim)

        if self.scale_factor > 1.0:
            mscale = 0.1 * math.log(self.scale_factor) + 1.0
            d_half = self.head_dim // 2

            low = (d_half * math.log(self.org_context_length / (beta * 2 * math.pi)) / math.log(base))
            high = (d_half * math.log(self.org_context_length / (alpha * 2 * math.pi)) / math.log(base))

            assert 0 < low < high < d_half - 1, "Cutoffs mismatch: 0 < low < high < d_half - 1"

            interpolation = 1.0 / (self.scale_factor * pos_freqs)
            extrapolation = 1.0 / pos_freqs

            ramp = (torch.arange(d_half, dtype=torch.float, device=pos_freqs.device) - low) / (high - low)
            m = 1 - ramp.clamp(0, 1)

            inv_freq = (interpolation * (1 - m)) + (extrapolation * m)
        else:
            inv_freq = 1.0 / pos_freqs
            mscale = 1.0

        t = torch.arange(self.extended_context_length, dtype=torch.float32, device=pos_freqs.device)
        freqs = torch.outer(t, inv_freq)

        emb = torch.repeat_interleave(freqs, 2, dim=-1)
        cos = emb.cos() * mscale
        sin = emb.sin() * mscale

        return cos, sin

    def rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)

    def _apply_yarn(self, x: torch.Tensor, start_pos: int = 0) -> torch.Tensor:
        T = x.size(2)
        cos = self.cos[start_pos : start_pos + T].unsqueeze(0).unsqueeze(1).to(x.dtype)
        sin = self.sin[start_pos : start_pos + T].unsqueeze(0).unsqueeze(1).to(x.dtype)
        return (x * cos) + (self.rotate_half(x) * sin)

    def forward(self, x: torch.Tensor, use_cache: bool = False, start_pos: int = 0, return_attn: bool = False):
        B, T, C = x.shape

        q = self.W_query(x).view(B, T, self.num_heads, self.head_dim)
        k = self.W_key(x).view(B, T, self.num_heads, self.head_dim)
        v = self.W_value(x).view(B, T, self.num_heads, self.head_dim)

        q = self.q_norm(q)
        k = self.k_norm(k)

        q = q.transpose(1, 2).contiguous()
        k = k.transpose(1, 2).contiguous()
        v = v.transpose(1, 2).contiguous()

        q_rope = self._apply_yarn(q, start_pos).contiguous()
        k_rope = self._apply_yarn(k, start_pos).contiguous()

        if use_cache:
            if self.cache_k is None:
                self._init_cache(B, x.device, k_rope.dtype)

            end_pos = start_pos + T
            self.cache_k[:B, :, start_pos:end_pos, :] = k_rope
            self.cache_v[:B, :, start_pos:end_pos, :] = v

            k_attn = self.cache_k[:B, :, :end_pos, :]
            v_attn = self.cache_v[:B, :, :end_pos, :]
        else:
            k_attn = k_rope
            v_attn = v

        attn_scores = q_rope @ k_attn.transpose(2, 3) / (k_attn.shape[-1] ** 0.5)

        if T > 1:
            kv_seq_len = k_attn.size(2)
            mask = torch.ones(T, kv_seq_len, device=x.device, dtype=torch.bool)
            mask = torch.triu(mask, diagonal=kv_seq_len - T + 1)
            attn_scores = attn_scores.masked_fill_(mask, -torch.inf)

        attn_weights = torch.softmax(attn_scores, dim=-1)

        if self.training and self.dropout.p > 0.0:
            attn_weights = self.dropout(attn_weights)

        out = attn_weights @ v_attn
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_out(out)

        return (out, attn_weights) if return_attn else out

    def reset_cache(self):
        self.cache_k, self.cache_v = None, None

# -----------------------------------------------------------------------------
# Transformer Blocks & Main Model
# -----------------------------------------------------------------------------

class TransformerBlock(nn.Module):
    """A single Transformer block (Pre-LN architecture) containing Attention and FFN."""
    def __init__(self, args: ModelArgs):
        super().__init__()

        self.norm1 = nn.RMSNorm(args.d_model, eps=1e-6)
        self.attn = Attention(args)
        self.dropout = nn.Dropout(args.dropout)
        self.norm2 = nn.RMSNorm(args.d_model, eps=1e-6)

        self.is_moe = args.moe_args is not None

        if self.is_moe:
            self.ff = MoEFeedForward(args.moe_args, args.d_model, args.moe_args.hidden_dim_multplier_perexp * args.d_model) 
        else:
            self.ff = FeedForward_Gelu(args.d_model, args.hidden_dim_multiplier * args.d_model)  # FeedForwardSwiGLU 

        self.init_weights()

    def init_weights(self):
        nn.init.xavier_uniform_(self.attn.W_query.weight)
        nn.init.xavier_uniform_(self.attn.W_key.weight)
        nn.init.xavier_uniform_(self.attn.W_value.weight)
        nn.init.xavier_uniform_(self.attn.W_out.weight)

    def forward(self, x: torch.Tensor, use_cache: bool = False, start_pos: int = 0, return_attn: bool = False):
        normalized_1 = self.norm1(x)

        if return_attn:
            attn_out, attn_weights = self.attn(normalized_1, use_cache=use_cache, start_pos=start_pos, return_attn=True)
        else:
            attn_out = self.attn(normalized_1, use_cache=use_cache, start_pos=start_pos)
            attn_weights = None

        x = x + self.dropout(attn_out)

        normalized_2 = self.norm2(x)

        if self.is_moe:
            ff_out, aux_loss = self.ff(normalized_2)
        else:
            ff_out = self.ff(normalized_2)
            aux_loss = 0.0

        x = x + self.dropout(ff_out)

        return (x, attn_weights, aux_loss) if return_attn else (x, aux_loss)


class MixGPTModel(nn.Module):
    """The main language model combining token embedding, blocks, and language modeling head."""

    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.tok_emb = nn.Embedding(args.vocab_size, args.d_model)

        if args.moe_args:
            print("Using MoE...")
        else:
            print("Using FF...")

        self.blocks = nn.ModuleList([TransformerBlock(args) for _ in range(args.num_layers)])

        self.norm_final = nn.RMSNorm(args.d_model, eps=1e-6)
        self.head = nn.Linear(args.d_model, args.vocab_size, bias=False)

    def forward(self, idx: torch.Tensor, targets: Optional[torch.Tensor] = None,
                use_cache: bool = False, start_pos: int = 0, return_attn: bool = False):

        x = self.tok_emb(idx)
        last_weights = None
        total_aux_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype) if self.args.moe_args else 0.0

        for block in self.blocks:
            if return_attn:
                x, last_weights, aux_loss = block(x, use_cache=use_cache, start_pos=start_pos, return_attn=True)
            else:
                x, aux_loss = block(x, use_cache=use_cache, start_pos=start_pos)

            if self.args.moe_args:
                total_aux_loss += aux_loss.to(x.dtype)

        x = self.norm_final(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            logits_flat = logits.view(-1, logits.size(-1))
            targets_flat = targets.view(-1)

            main_loss = F.cross_entropy(logits_flat, targets_flat)
            loss = main_loss + total_aux_loss

        return (logits, loss, last_weights) if return_attn else (logits, loss)

    def reset_kv_cache(self):
        """Flushes KV states entirely for new inference instances."""
        for blk in self.blocks:
            blk.attn.reset_cache()


In [ ]:

# -----------------------------------------------------------------------------
# Data Processing & Utilities
# -----------------------------------------------------------------------------

def get_tokenizer(tokenizer_path: str):
    """Loads the custom HuggingFace tokenizer from the specified absolute path."""
    src_model = os.path.abspath(tokenizer_path)
    return AutoTokenizer.from_pretrained(src_model)

def get_batch(tokens: torch.Tensor, context_length: int, batch_size: int, device: torch.device):
    """Samples a random batch of token sequences and their targets."""
    max_idx = len(tokens) - context_length - 1
    ix = torch.randint(0, max_idx, (batch_size,)).tolist()

    x = torch.stack([tokens[i : i + context_length] for i in ix]).to(device)
    y = torch.stack([tokens[i + 1 : i + context_length + 1] for i in ix]).to(device)

    return x, y

def get_lr(current_step: int, max_steps: int, base_lr: float, warmup_steps: int = 100, min_lr: float = 1e-5) -> float:
    """Calculates the learning rate with linear warmup and cosine decay."""
    if current_step < warmup_steps:
        return base_lr * (current_step + 1) / warmup_steps
    if current_step > max_steps:
        return min_lr

    decay_ratio = (current_step - warmup_steps) / (max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (base_lr - min_lr)

def calculate_active_params_MoE(model):
    """Calculates total and active parameters for a model using your custom

    MoEFeedForward layers.
    """
    total_params = 0
    total_expert_params = 0

    # Tracks the top_k settings from your code
    num_experts = None
    top_k = None

    # 1. Loop through all parameters in the model
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        num_el = param.numel()
        total_params += num_el

        # Check if the parameter belongs to your expert weights
        if any(w in name for w in [".w1", ".w2", ".w3"]):
            total_expert_params += num_el

    # 2. Find the MoE configuration dynamically from the layer
    for module in model.modules():
        if module.__class__.__name__ == "MoEFeedForward":
            num_experts = module.num_experts
            top_k = module.num_experts_per_tok
            break  # Assume all MoE layers use the same args

    # Safety check if no MoE layer was found
    if num_experts is None or top_k is None:
        raise ValueError("Could not find any MoEFeedForward layers in the model.")

    # 3. Math to find active parameters
    # Divide total expert parameters by the number of experts to find 1 expert's size
    single_expert_size = total_expert_params // num_experts

    # Subtract all experts, then add back only the active ones per token
    base_params = total_params - total_expert_params
    active_params = base_params + (single_expert_size * top_k)

    return {
        "total_parameters": total_params,
        "active_parameters": active_params,
        "shared_parameters": base_params,
    }



In [ ]:
## Train

torch.cuda.empty_cache()
torch.manual_seed(111)

paths = ProjectPaths()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Starting training on device: {device}")

os.makedirs(paths.model_checkpoints_dir, exist_ok=True)

# 1. Prepare Data
tokenizer = get_tokenizer(paths.tokenizer_dir)
with open(paths.data_corpus, "r", encoding="utf-8") as f:
    text = f.read()

ids = tokenizer.encode(text)
tokens = torch.tensor(ids, dtype=torch.long)
safe_vocab_size = max(len(tokenizer), tokens.max().item() + 1)
print(f"Total tokens: {len(tokens)} | Safe Vocab Size: {safe_vocab_size}")

# 2. Define Configurations
train_args = TrainingArgs()
# moe_args = MoEArgs(num_experts=2, top_k_experts=1, hidden_dim_multplier_perexp=2)
moe_args = None # for Dense model training

# Model architecture blueprint
model_args = ModelArgs(
    vocab_size=safe_vocab_size,
    d_model=1024,
    num_heads=16,
    num_layers=8,
    hidden_dim_multiplier=6,
    org_context_length=1024,
    scale_factor=2.0,
    rotary_freq_base=10000,
    dropout=0.1,
    moe_args=moe_args,  # Comment this for Dense only
    device=device
)

# 3. Initialize Model & Optimizer
model = MixGPTModel(model_args).to(device)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

if moe_args:
    moe_param_count = calculate_active_params_MoE(model)
    print(f"Active: {moe_param_count['active_parameters']/1e6}M | Shared: {moe_param_count['shared_parameters']/1e6}M")

optimizer = torch.optim.AdamW(model.parameters(), lr=train_args.lr_rate)

# 4. Training Execution 
model.train() 
st_time = time.perf_counter()
net_step_time = 0.0

for step in range(train_args.tot_steps):
    t0 = time.perf_counter()

    lr = get_lr(step, train_args.tot_steps, train_args.lr_rate)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    x, y = get_batch(tokens, context_length=model_args.org_context_length, batch_size=train_args.batch_size, device=device)
    logits, loss = model(x, targets=y)

    loss = loss / train_args.grad_accum_steps
    loss.backward()

    if ((step + 1) % train_args.grad_accum_steps == 0) or (step + 1 == train_args.tot_steps):
        torch.nn.utils.clip_grad_norm_(model.parameters(), train_args.grad_clip)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    if step >= (train_args.tot_steps - train_args.batch_size):
        preds = torch.argmax(logits, dim=-1)
        correct_mask = (preds == y)
        total_correct = correct_mask.sum().item()
        total_tokens = y.numel()
        print(f"Step:{step} | Accuracy: {(total_correct/total_tokens) * 100}")

    step_time = time.perf_counter() - t0
    net_step_time += step_time

    if step % 10 == 0:
        avg_step_time = net_step_time / 10 if step > 0 else step_time
        print(f"Step: {step:4d} | Loss: {loss.item()} | Avg Step Time: {avg_step_time:.4f}s")
        net_step_time = 0.0

tot_time = time.perf_counter() - st_time
last_loss = loss.item()
print(f"\nTraining execution loop completed successfully in: {tot_time:.5f}s")

# 5. Save Artifacts and Configuration (YAML)
checkpoint_path = os.path.join(paths.model_checkpoints_dir, 'gpt_moe_production.pt')
config_path = os.path.join(paths.model_checkpoints_dir, 'training_config_production.yaml')

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict()
}, checkpoint_path)

training_params = {
    'hyperparameters': {
        'vocab_size': model_args.vocab_size,
        'model_dim': model_args.d_model,
        'num_heads': model_args.num_heads,
        'num_layers': model_args.num_layers,
        'ff_hidden_dim_multiplier': model_args.hidden_dim_multiplier,
        'org_context_length': model_args.org_context_length,
        'scale_factor': model_args.scale_factor,
        'rotary_freq_base': model_args.rotary_freq_base,
        'dropout': model_args.dropout,
        'moe_config': {
            'num_experts': moe_args.num_experts,
            'top_k_experts': moe_args.top_k_experts,
            'hidden_dim_multiplier_per_expert': moe_args.hidden_dim_multplier_perexp,
        } if moe_args else None,
    },
    'training_stats': {
        'tot_steps': train_args.tot_steps,
        'batch_size': train_args.batch_size,
        'loss': float(f"{last_loss:.6f}"),
        'training_time_s': float(f"{tot_time:.4f}")
    }
}

with open(config_path, 'w') as file:
    yaml.dump(training_params, file)

print(f"\nCheckpoint saved to {checkpoint_path}")
print(f"Config saved to {config_path}")

del model

# MoE => (Total parameters: 138.78M | Active:88.45M)
"""
Step:  490 | Loss: 0.1013 | Avg Step Time: 0.5701s
Step:498 | Accuracy: 99.560546875
Step:499 | Accuracy: 99.462890625
Training execution loop completed successfully in: 285.35194s
"""

# Dense => (Total parameters: 138.77M)
"""
Step:  490 | Loss: 0.03539933264255524 | Avg Step Time: 0.6355s
Step:498 | Accuracy: 99.560546875
Step:499 | Accuracy: 99.51171875
Training execution loop completed successfully in: 352.69447s
"""


### Inference


In [ ]:
## Init

torch.cuda.empty_cache()

@torch.no_grad()
def generate_minP(model: MixGPTModel, tokenizer, prompt: str, device: torch.device,
                  max_new_tokens: int = 100, temp: float = 1.0, min_p: float = 0.1, use_cache: bool = True):
    """Autoregressive text generation utilizing Min-P sampling."""
    model.eval()
    encoded_token_ids = tokenizer.encode(prompt)
    idx = torch.tensor([encoded_token_ids], dtype=torch.long).to(device)
    temperature = max(temp, 1e-5)

    if use_cache:
        model.reset_kv_cache()
        start_pos = 0

        # Initial prompt processing
        logits, _ = model(idx, use_cache=True, start_pos=start_pos)
        next_token_logits = logits[:, -1, :] / temperature

        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        generated_tokens = [next_token]

        # Autoregressive Phase
        for _ in range(max_new_tokens - 1):
            start_pos = idx.size(1) + len(generated_tokens) - 1
            logits, _ = model(next_token, use_cache=True, start_pos=start_pos)

            next_token_logits = logits[:, -1, :] / temperature
            probs = torch.softmax(next_token_logits, dim=-1)

            # Min-P Filtering
            p_max, _ = probs.max(dim=-1, keepdim=True)
            threshold = p_max * min_p
            filtered_probs = torch.where(probs >= threshold, probs, torch.zeros_like(probs))

            # Renormalize and sample
            denom = filtered_probs.sum(dim=-1, keepdim=True)
            filtered_probs = filtered_probs / (denom + 1e-8)
            next_token = torch.multinomial(filtered_probs, num_samples=1)

            generated_tokens.append(next_token)

        return torch.cat([idx] + generated_tokens, dim=1)

    else:
        # Non-cached generation (Slower)
        for _ in range(max_new_tokens):
            logits, _ = model(idx)
            curr_logits = logits[:, -1, :] / temperature
            probs = torch.softmax(curr_logits, dim=-1)

            p_max, _ = probs.max(dim=-1, keepdim=True)
            threshold = p_max * min_p
            filtered_probs = torch.where(probs >= threshold, probs, torch.zeros_like(probs))

            denom = filtered_probs.sum(dim=-1, keepdim=True)
            filtered_probs = filtered_probs / (denom + 1e-8)
            next_token = torch.multinomial(filtered_probs, num_samples=1)

            idx = torch.cat((idx, next_token), dim=1)

        return idx


paths = ProjectPaths()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(os.path.abspath(paths.tokenizer_dir))

# 2. Load Configuration Dynamically
config_path = os.path.join(paths.model_checkpoints_dir, 'training_config_production.yaml')
checkpoint_path = os.path.join(paths.model_checkpoints_dir, 'gpt_moe_production.pt')

if not os.path.exists(config_path):
    raise FileNotFoundError(f"Configuration file not found at {config_path}. Did you run train.py first?")

with open(config_path, 'r') as file:
    tr_params = yaml.safe_load(file)

hp = tr_params['hyperparameters']

# Allows scaling the context length independently during inference
og_scale_factor = hp['scale_factor']
new_scale_factor = 1.0  # Adjust this if you want to extend context further during generation
final_scale_factor = og_scale_factor * new_scale_factor

print(f"Original Context: {hp['org_context_length']} | Final Inference Scale: {final_scale_factor}")

moe_args = None
if hp.get('moe_config'):
    moe_args = MoEArgs(
        num_experts=hp['moe_config']['num_experts'],
        top_k_experts=hp['moe_config']['top_k_experts'],
        hidden_dim_multplier_perexp=hp['moe_config']['hidden_dim_multiplier_per_expert']
    )

model_args = ModelArgs(
    vocab_size=hp['vocab_size'],
    d_model=hp['model_dim'],
    num_heads=hp['num_heads'],
    num_layers=hp['num_layers'],
    hidden_dim_multiplier=hp["ff_hidden_dim_multiplier"],
    org_context_length=hp['org_context_length'],
    scale_factor=final_scale_factor,
    rotary_freq_base=hp['rotary_freq_base'],
    dropout=hp['dropout'],
    moe_args=moe_args,
    device=device
)

# 3. Load Model State
model = MixGPTModel(model_args)

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    print(f"Successfully loaded checkpoint from {checkpoint_path}")
else:
    raise FileNotFoundError(f"Model checkpoint not found at {checkpoint_path}")

model.to(device)


In [ ]:
# 4. Execute Inference
prompt = "The cat (Felis"
gen_token_count = 512

print(f"\nGenerating with Min-P...")
gen_st_time = time.perf_counter()

token_ids = generate_minP(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt.strip(),
    device=device,
    max_new_tokens=gen_token_count,
    temp=0.5,
    min_p=0.1,
    use_cache=True
)

decoded_text = tokenizer.decode(token_ids[0].tolist())
tot_gen_time = time.perf_counter() - gen_st_time

resp_token_ids = tokenizer.encode(decoded_text)
print(f"Response Token len: {len(resp_token_ids)}")

print(f"Response: {decoded_text}\n")
print(f"Inference time: {tot_gen_time:.5f}s")
print(f"{int(len(token_ids[0])/tot_gen_time)} tokens/sec")
